# Chapter 15 — The Agent Writes Its Own Context

## Question

**What happens when the model becomes a producer of the information that later computations consume?**

Falsifiable structure: does persisting a hypothesis without status lengthen its live miscounted-as-fact lifetime versus persisting it with status, or not persisting it at all? Only observable application state appears below — plans, hypotheses, summaries, reports — never private reasoning traces.

## Setup — three producers, one lifecycle

EXTERNAL items report what the environment said. AGENT items report what the agent thinks. HARNESS items report what the runtime decided. Created, persisted, and admitted are observable; used, influential, and correct are explicitly NOT_MEASURED here.

In [ ]:
from dataclasses import dataclass, field
from enum import Enum

class Producer(Enum):
    EXTERNAL = 'EXTERNAL'
    AGENT = 'AGENT'
    HARNESS = 'HARNESS'

class Status(Enum):
    UNVERIFIED = 'UNVERIFIED'
    CONFIRMED = 'CONFIRMED'
    FALSIFIED = 'FALSIFIED'

@dataclass
class StateItem:
    id: str
    kind: str
    producer: Producer
    text: str
    status: Status = None
    evidence_refs: tuple = ()
    confirming_evidence_id: str = None
    tokens: int = 0
    lifecycle: str = 'CREATED'  # CREATED | PERSISTED | ADMITTED
    used: str = 'NOT_MEASURED'
    influential: str = 'NOT_MEASURED'
    correct: str = 'NOT_MEASURED'

obs = StateItem('obs-17', 'observation', Producer.EXTERNAL, 'FAIL: expected 4, got 5', tokens=30)
hyp = StateItem('hyp-3', 'hypothesis', Producer.AGENT, 'Cache invalidation is broken',
                  status=Status.UNVERIFIED, evidence_refs=('obs-17',), tokens=40)
print(f'observation: {obs.text!r} (producer={obs.producer.value})')
print(f'hypothesis:  {hyp.text!r} (status={hyp.status.value}, evidence={hyp.evidence_refs})')
assert hyp.status == Status.UNVERIFIED and hyp.evidence_refs

## Baseline — a five-step trajectory with a wrong guess

Step 1 observes a failure. Step 2 hypothesises cache invalidation. Step 3 persists. Step 4 re-admits. Step 5 brings counter-evidence: the serializer, not the cache.

In [ ]:
STEPS = ['observation: FAIL expected 4 got 5', 'hypothesis: cache invalidation broken',
         'persisted state written', 'state re-admitted', 'counter-evidence: serializer fault']
print(' -> '.join(f'step{i+1}' for i in range(len(STEPS))))
assert len(STEPS) == 5

## Intervention — five persistence conditions, structural metrics only

A appears once, never persists. B persists without status. C persists with UNVERIFIED status plus evidence reference. D receives counter-evidence but never updates old state. E invalidates explicitly on contradiction. Metrics count readmissions and live steps; no behavioural harm is claimed.

In [ ]:
RESULTS = {
    'A once-never-persisted': {'readmissions': 0, 'steps_live': 1, 'correction_delay': 0, 'contradictory': 0},
    'B persisted-no-status': {'readmissions': 4, 'steps_live': 4, 'correction_delay': 4, 'contradictory': 1},
    'C persisted-with-status': {'readmissions': 4, 'steps_live': 4, 'correction_delay': 1, 'contradictory': 0},
    'D counterevidence-no-update': {'readmissions': 5, 'steps_live': 5, 'correction_delay': 5, 'contradictory': 2},
    'E explicit-invalidation': {'readmissions': 2, 'steps_live': 2, 'correction_delay': 1, 'contradictory': 0},
}
print(f"{'condition':28s} {'readmit':>7s} {'live':>4s} {'delay':>5s} {'contra':>6s}")
for name, m in RESULTS.items():
    print(f"{name:28s} {m['readmissions']:7d} {m['steps_live']:4d} {m['correction_delay']:5d} {m['contradictory']:6d}")
assert RESULTS['B persisted-no-status']['contradictory'] > RESULTS['C persisted-with-status']['contradictory']
assert RESULTS['E explicit-invalidation']['steps_live'] < RESULTS['D counterevidence-no-update']['steps_live']
print('Persistence amplifies the mistake and the correction alike: structure decides which.')

## Versioned state — confirmation needs backing

In [ ]:
v1 = StateItem('hyp-3', 'hypothesis', Producer.AGENT, 'cache invalidation broken', Status.UNVERIFIED, ('obs-17',))
v2 = StateItem('hyp-3', 'hypothesis', Producer.AGENT, 'cache invalidation broken', Status.FALSIFIED, ('obs-17', 'obs-31'))
v3 = StateItem('hyp-3', 'hypothesis', Producer.AGENT, 'serializer corrupts timestamps', Status.CONFIRMED, ('obs-31',),
                  confirming_evidence_id='obs-31')
print('v1 UNVERIFIED -> v2 FALSIFIED -> v3 CONFIRMED (backed by obs-31)')
assert v3.confirming_evidence_id is not None
try:
    bare = StateItem('hyp-9', 'hypothesis', Producer.AGENT, 'it was the cache', Status.CONFIRMED)
    assert bare.confirming_evidence_id is not None, 'confirmed with no evidence is a self-awarded win'
except AssertionError as e:
    print(f'refused: {e}')

## Free-form versus bounded structured state

In [ ]:
freeform = {'tokens': 600, 'duplicates': 3, 'status_visible': False, 'replaceable': False, 'invalidated_still_live': 2}
structured = {'goal': 'fix migration test', 'confirmed_facts': ['obs-17'], 'open_hypotheses': ['hyp-3?'],
              'decisions': [], 'remaining_work': ['reproduce'], 'evidence_refs': ['obs-17'],
              'tokens': 380, 'duplicates': 0, 'status_visible': True, 'replaceable': True, 'invalidated_still_live': 0}
print(f"free-form:  {freeform['tokens']} tokens, status visible={freeform['status_visible']}, stale-live={freeform['invalidated_still_live']}")
print(f"structured: {structured['tokens']} tokens, status visible={structured['status_visible']}, stale-live={structured['invalidated_still_live']}")
assert structured['tokens'] < freeform['tokens']
print('Fixture-specific structural properties only: no claim that structured state wins in production.')

## Traps — append-only structure, path dependence, handoff

In [ ]:
# Append-only trap: v1 active and v2 false both rendered live.
live_render = ['v1: hypothesis active', 'v2: hypothesis false']
print(f'append-only live render holds {len(live_render)} contradictory states')
assert len(live_render) == 2
print('Structured does not automatically mean bounded or correct.')

# Path dependence: identical Context_0, different first observations.
universe_a = {'cache-search-results', 'invalidation-hypothesis'}
universe_b = {'migration-log', 'timeout-hypothesis'}
print(f'same Context_0 -> universes differ: {universe_a != universe_b}')
assert universe_a != universe_b

# Cross-agent handoff: report with status versus bare verdict.
report = {'producer': 'subagent-2', 'status': 'HYPOTHESIS', 'evidence': 'observation-17',
          'text': 'Migration definitely failed because X'}
print(f"handoff keeps producer={report['producer']}, status={report['status']}, evidence={report['evidence']}")
assert report['status'] == 'HYPOTHESIS'
print('A bare verdict would travel as fact; the record travels as hypothesis.')

## Try it

1. Drop `evidence_refs` from condition C and confirm it becomes structurally identical to B.
2. Add a v4 that re-confirms v3 without new evidence and watch the backing assertion refuse it.
3. Render only v2 (never v1) and recount live contradictions: invalidation works when the renderer cooperates.

In [ ]:
# Reader scratch space (commented out so Run All stays at baseline):
# print(RESULTS['B persisted-no-status'])
# print(StateItem('h', 'hypothesis', Producer.AGENT, 'x').status)  # None: statusless

## What this demonstrates

- Execution produces future context candidates: the candidate universe does not stand still.
- Persisting an interpretation without its epistemic status leaves later context unable to distinguish observation from hypothesis.
- Persistence amplifies both useful state and mistakes; readmission counts make the amplification visible.

## What this does not demonstrate

- That structured state improves real-agent success, or that plans are always useful.
- That agent-generated state should be removed, or that uncertainty tags change model behaviour.
- That persistence necessarily causes error, or that influence was measured here.
- Anything about hidden chain-of-thought: only observable application state is governed.

## Connection to the chapter

When useful state outlives its trajectory, it crosses from working state into memory candidacy:

> When useful state survives beyond the current trajectory for reuse by future tasks, it crosses the boundary from working state into memory candidacy.

That is Chapter 16.